#### Which are the main countries in which textiles are produced that the public authorities in Germany buy?

In [1]:
import os
import re
from fuzzywuzzy import process
import pycountry
from collections import Counter
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd


In [2]:
def get_filenames(directory):
    """
    Return a list of filenames inside the directory.
    """
    return [f for f in os.listdir(directory) 
            if os.path.isfile(os.path.join(directory, f))]


In [3]:

def normalize_country(name, threshold=85):
    """
    Fully automatic normalisation of any token ~> country name using fuzzy matching only.
    """
    if not isinstance(name, str):
        return None

    name = name.strip()

    # All valid country names
    all_countries = [c.name for c in pycountry.countries]

    # Fuzzy match any token
    best, score = process.extractOne(name, all_countries)

    if score >= threshold:
        return best
    return None


def extract_countries_from_filename(filename):
    """
    Extracts countries using only fuzzy matching against pycountry.
    No skip-lists, no manual rules.
    """
    base = os.path.splitext(filename)[0]

    # Replace delimiters with spaces
    cleaned = re.sub(r"[^\w]", " ", base)  # keep only letters/numbers
    tokens = cleaned.split()

    detected = set()

    for token in tokens:
        if len(token) < 3:
            continue
        country = normalize_country(token)
        if country:
            detected.add(country)

    return list(detected)


def get_country_counts_from_filenames(file_list):
    counter = Counter()

    for fname in file_list:
        countries = extract_countries_from_filename(fname)
        counter.update(countries)

    return dict(counter)



In [ ]:
folder = "tradeatlas_files"
file_list = get_filenames(folder)

country_counts = get_country_counts_from_filenames(file_list)
print(country_counts)

{'Pakistan': 2, 'Russian Federation': 1, 'Bangladesh': 1, 'China': 1, 'Malaysia': 1}


In [5]:
def get_tradeatlas_export_counts(directory):
    counter = Counter()

    for f in os.listdir(directory):
        if f.lower().endswith((".xlsx",".csv")):
            df = pd.read_excel(os.path.join(directory, f))

            # find country-like column
            possible_cols = [c for c in df.columns if "country" in c.lower()]
            if not possible_cols:
                continue

            col = possible_cols[0]
            
            for item in df[col].dropna().astype(str):
                c = normalize_country(item)
                if c:
                    counter[c] += 1

    return dict(counter)


In [6]:
def combine_everything(cir_dict, brand_dict, trade_dict):
    combined = Counter()
    combined.update(cir_dict)
    combined.update(brand_dict)
    combined.update(trade_dict)

    df = pd.DataFrame(
        [{"country": c, "count": n} for c, n in combined.items()]
    )
    
    # keep all rows regardless of ISO3 success
    df = df.sort_values("count", ascending=False).reset_index(drop=True)
    
    return df


In [7]:
def plot_import_map(df):
    # add ISO3 codes
    df["iso3"] = df["country"].apply(
        lambda x: pycountry.countries.get(name=x).alpha_3 
        if pycountry.countries.get(name=x) else None
    )

    fig = px.scatter_geo(
        df,
        locations="iso3",
        size="count",
        hover_name="country",
        color="count",
        projection="natural earth",
        title="Combined Import Country Map (CIR + Brand Files + TradeAtlas)"
    )
    
    fig.update_geos(showcountries=True)
    fig.show()


In [8]:
cir_counts = {
    # ---- ASIA ----
    "China": 13,
    "Turkey": 12,
    "Bangladesh": 11,
    "Pakistan": 11,
    "Vietnam": 11,
    "India": 7,
    "Laos": 7,
    "Cambodia": 4,
    "Uzbekistan": 4,
    "Indonesia": 3,
    "Myanmar": 3,
    "Sri Lanka": 3,
    "Malaysia": 2,
    "Hong Kong": 2,
    "South Korea": 2,
    "Taiwan": 2,
    "Japan": 1,
    "Lebanon": 1,
    "Singapore": 1,

    # ---- SOUTH AMERICA ----
    "Mexico": 3,
    "Brazil": 2,

    # ---- SOUTH / EASTERN EUROPE ----
    "North Macedonia": 9,
    "Albania": 7,
    "Bulgaria": 6,
    "Romania": 6,
    "Bosnia and Herzegovina": 5,
    "Poland": 5,
    "Ukraine": 5,
    "Armenia": 3,
    "Croatia": 3,
    "Moldova": 3,
    "Slovakia": 3,
    "Czech Republic": 3,
    "Hungary": 3,
    "Latvia": 2,
    "Lithuania": 2,
    "Slovenia": 2,
    "Estonia": 1,
    "Serbia": 1,

    # ---- AFRICA ----
    "Tunisia": 15,
    "Ethiopia": 3,
    "Morocco": 3,
    "Egypt": 1,
    "Mauritius": 1,
    "Zimbabwe": 1,

    # ---- EUROPE (WEST/NORTH) ----
    "Germany": 17,
    "Italy": 6,
    "Spain": 5,
    "France": 4,
    "Portugal": 4,
    "Denmark": 2,
    "Austria": 2,
    "Sweden": 2,
    "Belgium": 1,
    "United Kingdom": 1
}


In [ ]:
trade_dir = "tradeatlas_files"  # update when ready

# ------ PROCESS ------
brand_counts = get_country_counts_from_filenames(file_list=file_list)
trade_counts = get_tradeatlas_export_counts(trade_dir)

df_final = combine_everything(cir_counts, brand_counts, trade_counts)


In [ ]:

SPECIAL_ISO3 = {
    "Viet Nam": "VNM",
    "Vietnam": "VNM",
    "Russian Federation": "RUS",
    "Russia": "RUS",
    "Czechia": "CZE",
    "Ivory Coast": "CIV",
    "Côte d’Ivoire": "CIV",
    "Cote d'Ivoire": "CIV",
    "South Korea": "KOR",
    "Republic of Korea": "KOR",
    "North Macedonia": "MKD",
    "Türkiye": "TUR",
    "Turkey": "TUR",
}

def iso3_from_name(name: str):
    if pd.isna(name):
        return None
    name = str(name).strip()
    if name in SPECIAL_ISO3:
        return SPECIAL_ISO3[name]
    try:
        return pycountry.countries.lookup(name).alpha_3
    except LookupError:
        return None


def plot_germany_flow_map(df_final, html_path=None):
    """
    Dark flow map from Germany to all countries in df_final.
    If html_path is given, saves the figure as an HTML file.
    """
    df = df_final.copy()

    df["iso3"] = df["country"].apply(iso3_from_name)
    missing = df[df["iso3"].isna()]["country"].unique().tolist()
    if missing:
        print("⚠ Could not map these countries to ISO3 (will be skipped):", missing)

    df = df.dropna(subset=["iso3", "count"])

    df_targets = df[df["iso3"] != "DEU"]

    colorscale = ["#6a8bff", "#ff4ecb"]
    min_count = df_targets["count"].min()
    max_count = df_targets["count"].max()

    def line_color(value):
        t = (value - min_count) / (max_count - min_count + 1e-9)
        return colorscale[0] if t < 0.5 else colorscale[1]

    def line_width(value):
        return 1.0 + 3.0 * (value - min_count) / (max_count - min_count + 1e-9)

    fig = go.Figure()

    # lines from Germany
    for _, row in df_targets.iterrows():
        fig.add_trace(go.Scattergeo(
            locationmode="ISO-3",
            mode="lines",
            locations=["DEU", row["iso3"]],
            line=dict(
                width=line_width(row["count"]),
                color=line_color(row["count"])
            ),
            opacity=0.75,
            hoverinfo="text",
            text=(
                f"Germany (DEU) → {row['country']} ({row['iso3']})<br>"
                f"Count: {row['count']}"
            ),
            showlegend=False
        ))

    # Germany node
    fig.add_trace(go.Scattergeo(
        locationmode="ISO-3",
        locations=["DEU"],
        mode="markers+text",
        marker=dict(
            size=18,
            color="#00d7ff",
            line=dict(color="white", width=1.2),
            opacity=1.0
        ),
        text=["GERMANY"],
        textposition="bottom center",
        textfont=dict(color="white", size=12, family="Arial"),
        hoverinfo="skip",
        name="Germany"
    ))

    # destination nodes
    fig.add_trace(go.Scattergeo(
        locationmode="ISO-3",
        locations=df_targets["iso3"],
        mode="markers+text",
        marker=dict(
            size=(df_targets["count"] / df_targets["count"].max()) * 10 + 6,
            color="#ff4ecb",
            line=dict(color="white", width=0.7),
            opacity=0.95
        ),
        text=df_targets["iso3"],
        textposition="top center",
        textfont=dict(color="white", size=9, family="Arial"),
        hoverinfo="skip",
        name="Destinations"
    ))

    # side legend
    sorted_list = (
        df_targets.sort_values("count", ascending=False)
        .apply(lambda r: f"{r['iso3']}: {r['count']}", axis=1)
        .tolist()
    )
    legend_text = "<br>".join(sorted_list)

    fig.update_layout(
        annotations=[
            dict(
                x=1.05,
                y=0.5,
                xref="paper",
                yref="paper",
                text=f"<b>Destinations & Counts</b><br><br>{legend_text}",
                showarrow=False,
                font=dict(size=12, color="white"),
                align="left",
            )
        ]
    )

    fig.update_layout(
        geo=dict(
            projection_type="natural earth",
            showland=True,
            landcolor="#05070c",
            countrycolor="#444",
            coastlinecolor="#444",
            showcountries=True,
            showframe=False,
            showocean=True,
            oceancolor="#02040a",
        ),
        paper_bgcolor="#000000",
        plot_bgcolor="#000000",
        height=850,
        width=1900,
        margin=dict(l=0, r=250, t=80, b=0),
        showlegend=False
    )


    if html_path is not None:
        # Create folder if missing
        os.makedirs(os.path.dirname(html_path), exist_ok=True)


        fig.write_html(html_path, include_plotlyjs="cdn")
        print(f"Saved interactive map to: {html_path}")



In [ ]:
# df_final must have at least: 'country' and 'count'
# 1. Generate and save the map as usual
fig = plot_germany_flow_map(df_final, html_path=r"outputs/germany_flows.html")

# 2. Force Plotly to display the interactive graph
fig.show()

Saved interactive map to: outputs/germany_flows.html
